# Epi Info AI Population Survey validation lab — V0.12

Validate the candidate `epi.sampleSize.populationSurvey` Rust/WebAssembly kernel against the audited legacy algorithm and an ordinary SciPy normal-quantile comparison. Passing is evidence, not statistical approval; G5 remains consolidated.

In [ ]:
import math
from pyodide.http import pyfetch
from js import WebAssembly, Uint8Array
from scipy.stats import norm
fixture_response = await pyfetch('../../validation-fixtures/population-survey-v0.12.json')
fixture_response.raise_for_status(); fixture = await fixture_response.json()
wasm_response = await pyfetch('../../epi2x2.wasm')
instance = await WebAssembly.instantiate(Uint8Array.new(await wasm_response.buffer()), {})
rust = instance.instance.exports


In [ ]:
def legacy_norm_tail(z):
    z = abs(z); p = 1 + z * (0.04986735 + z * (0.02114101 + z * (0.00327763 + z * (0.0000380036 + z * (0.0000488906 + z * 0.000005383)))))
    p = p*p; p = p*p; p = p*p; return 1/(p*p)
def legacy_anorm(p):
    v = dv = .5; z = 0
    while dv > 1e-6:
        z = 1/v - 1; dv /= 2
        v = v-dv if legacy_norm_tail(z) > p else v+dv
    return z
def legacy_cluster_size(i, level):
    factor = i['expectedFrequencyPercent'] * (100-i['expectedFrequencyPercent']) / i['marginOfErrorPercent']**2
    n = legacy_anorm(1-level)**2 * factor; corrected = n/(1+n/i['populationSize'])
    return math.ceil(i['designEffect'] * round(corrected) / i['clusters'])
case = fixture['cases'][0]; levels = [row['confidenceLevel'] for row in case['expected']]
python_sizes = [legacy_cluster_size(case['input'], level) for level in levels]
assert python_sizes == [row['clusterSize'] for row in case['expected']]
print('PASS: independent Python translation matches all seven legacy default outputs')


In [ ]:
i = case['input']; factor = i['expectedFrequencyPercent']*(100-i['expectedFrequencyPercent'])/i['marginOfErrorPercent']**2
scipy_sizes = []
for level in levels:
    z = norm.ppf((1+level)/2); n = z*z*factor; scipy_sizes.append(round(n/(1+n/i['populationSize'])))
list(zip(levels, python_sizes, scipy_sizes))


In [ ]:
def wasm_size(i, level): return int(rust.population_survey_cluster_size(i['populationSize'], i['expectedFrequencyPercent'], i['marginOfErrorPercent'], i['designEffect'], i['clusters'], level))
wasm_sizes = [wasm_size(case['input'], level) for level in levels]
assert wasm_sizes == python_sizes
clustered = fixture['cases'][1]; expected95 = clustered['expected95']; size95 = wasm_size(clustered['input'], .95)
assert size95 == expected95['clusterSize']; assert size95*clustered['input']['clusters'] == expected95['totalSample']
assert math.isnan(float(rust.population_survey_cluster_size(0, 50, 5, 1, 1, .95)))
print('PASS: deployed Rust/WASM matches defaults, clustered design, and fail-closed boundary')
